In [1]:
from Common import Common
from Dyson import Dyson
from Bare import Bare
import numpy as np
import os, sys
qapath = os.environ.get('QAssemble')
sys.path.append(qapath+'/src/QAssemble/modules')
import QAFort

In [2]:
n = 10
for i in range(n):
    itheta = Common.Ttind(i, n)
    itheta2 = QAFort.common.ttind(i, n)
    print(i, n, itheta, itheta2)

0 10 9 9
1 10 8 8
2 10 7 7
3 10 6 6
4 10 5 5
5 10 4 4
6 10 3 3
7 10 2 2
8 10 1 1
9 10 0 0


In [2]:
norb = 4
ns = 2
nk = 5
nomega = 1000

tempmat1 = np.zeros([norb,norb],dtype=np.complex128,order='F')
tempmat2 = np.zeros([norb,norb],dtype=np.complex128,order='F')
tempmat3 = np.zeros([8,8],dtype=np.complex128,order='F')
tempmat4 = np.zeros([8,8],dtype=complex,order='F')
tempmat5 = np.zeros([8,8],dtype=complex,order='F')
tempmat6 = np.zeros([8,8],dtype=complex,order='F')
glatt0 = np.zeros([norb,norb,ns,nk,nomega],dtype=np.complex128,order='F')
glattref = np.zeros([norb,norb,ns,nk,nomega],dtype=np.complex128,order='F')
siglatt = np.zeros([norb,norb,ns,nk,nomega],dtype=np.complex128,order='F')
fhlatt = np.zeros([norb,norb,ns,nk],dtype=np.complex128,order='F')

omega = np.zeros([nomega],dtype=np.float64,order='F')

In [3]:
for iomega in range(nomega):
    omega[iomega] = (2*iomega+1)

for ik in range(nk):
    for js in range(ns):
        for iorb in range(norb):
            for jorb in range(norb):
                if iorb==jorb:
                    fhlatt[iorb,jorb,js,ik] = 1.0+(ik+1)+(js+1)*0.1
                else:
                    fhlatt[iorb,jorb,js,ik] = 0.1 +(ik+1)+(js+1)*0.1 + ((iorb+1)+(jorb+1))*0.1

for iomega in range(nomega):
    for ik in range(nk):
        for js in range(ns):
            for iorb in range(norb):
                for jorb in range(norb):
                    if iorb==jorb:
                        glatt0[iorb,jorb,js,ik,iomega] = 1.0/(omega[iomega]*1j-1.0 +(ik+1)+(js+1)*0.1 + ((iorb+1)+(jorb+1))*2.0)

for iomega in range(nomega):
    for ik in range(nk):
        for js in range(ns):
            for iorb in range(norb):
                for jorb in range(norb):
                    siglatt[iorb,jorb,js,ik,iomega] = 5*(ik+1)+(iorb+1)+0.1*(iomega+1)+(jorb+1)*2+(jorb+1)


            tempmat1 = Common.CmplxMatInv(glatt0[:,:,js,ik,iomega])
            tempmat2 = tempmat1-siglatt[:,:,js,ik,iomega]
            glattref[:,:,js,ik,iomega] = Common.CmplxMatInv(tempmat2)

In [4]:
glatdyn = np.zeros([norb,norb,ns,nk,nomega],dtype=complex,order='F')
glatdyn = Dyson.FLatDyn(glatt0, siglatt)

In [5]:
for iomega in range(nomega):
    for ik in range(nk):
        for js in range(ns):
            for jorb in range(norb):
                for iorb in range(norb):
                    err = glattref[iorb, jorb, js, ik, iomega] - glatdyn[iorb, jorb, js, ik, iomega]
                    if (abs(err) > 1.0e-6):
                        print(iorb, jorb, js, ik, iomega, abs(err), glattref[iorb, jorb, js, ik, iomega], glatdyn[iorb, jorb, js, ik, iomega])

In [6]:
import os, sys
qapath = os.environ.get('QAssemble')
sys.path.append(qapath+'/src/QAssemble/modules')
import QAFort

In [7]:
glatdyn2 = QAFort.dyson.flatdyn(glatt0, siglatt)

In [8]:
for iomega in range(nomega):
    for ik in range(nk):
        for js in range(ns):
            for jorb in range(norb):
                for iorb in range(norb):
                    err = glatdyn2[iorb, jorb, js, ik, iomega] - glatdyn[iorb, jorb, js, ik, iomega]
                    if (abs(err) > 1.0e-6):
                        print(iorb, jorb, js, ik, iomega, abs(err), glatdyn2[iorb, jorb, js, ik, iomega], glatdyn[iorb, jorb, js, ik, iomega])

In [9]:
ff = Bare.FLatFreq(omega, fhlatt)
ff2 = QAFort.bare.flatfreq(fhlatt, omega)

In [10]:
for iomega in range(nomega):
    for ik in range(nk):
        for js in range(ns):
            for jorb in range(norb):
                for iorb in range(norb):
                    err = ff2[iorb, jorb, js, ik, iomega] - ff[iorb, jorb, js, ik, iomega]
                    if (abs(err) > 1.0e-6):
                        print(iorb, jorb, js, ik, iomega, abs(err), ff2[iorb, jorb, js, ik, iomega], ff[iorb, jorb, js, ik, iomega])

In [11]:
ntau = nomega
tau = np.zeros([ntau],dtype=np.float64,order='F')
ftlatt = np.zeros([norb,norb,ns,nk,ntau],dtype=np.complex128,order='F')
ftlatt2 = np.zeros([norb,norb,ns,nk,ntau],dtype=np.complex128,order='F')

In [12]:
beta = 1.0/(8.617333262145e-5*300.0)
pi = np.pi
for itau in range(ntau):
    itheta = QAFort.common.ttind(itau,ntau)
    tau[itau] = beta/2.0*(np.cos(pi*(itheta+0.5)/ntau)+1)

fmoment, fhigh = QAFort.fourier.flatdyn_m(omega,ff2,1,1)

# ftlatt = QAFort.fourier.flatdyn_f2t(omega,ff2,fmoment,tau)

In [13]:
print(QAFort.fourier.flatdyn_f2t.__doc__)

ftau = flatdyn_f2t(omega,fomega,moment,tau,[norb,ns,nk,nomega,ntau])

Wrapper for ``flatdyn_f2t``.

Parameters
----------
omega : input rank-1 array('d') with bounds (nomega)
fomega : input rank-5 array('D') with bounds (norb,norb,ns,nk,nomega)
moment : input rank-5 array('D') with bounds (norb,norb,ns,nk,3)
tau : input rank-1 array('d') with bounds (ntau)

Other Parameters
----------------
norb : input int, optional
    Default: shape(fomega, 0)
ns : input int, optional
    Default: shape(fomega, 2)
nk : input int, optional
    Default: shape(fomega, 3)
nomega : input int, optional
    Default: shape(omega, 0)
ntau : input int, optional
    Default: shape(tau, 0)

Returns
-------
ftau : rank-5 array('D') with bounds (norb,norb,ns,nk,ntau)



In [14]:
ftlatt = QAFort.fourier.flatdyn_f2t(omega, ff2, fmoment, tau)

spreadcheck NU pt not in valid range (central three periods): kx[329]=9.469425314105727, N1=8000 (pirange=1)
FINUFFT invokeGuru: setpts error (ier=4)!
spreadcheck NU pt not in valid range (central three periods): kx[329]=9.469425314105727, N1=8000 (pirange=1)
FINUFFT invokeGuru: setpts error (ier=4)!
spreadcheck NU pt not in valid range (central three periods): kx[329]=9.469425314105727, N1=8000 (pirange=1)
FINUFFT invokeGuru: setpts error (ier=4)!
spreadcheck NU pt not in valid range (central three periods): kx[329]=9.469425314105727, N1=8000 (pirange=1)
FINUFFT invokeGuru: setpts error (ier=4)!
spreadcheck NU pt not in valid range (central three periods): kx[329]=9.469425314105727, N1=8000 (pirange=1)
FINUFFT invokeGuru: setpts error (ier=4)!
spreadcheck NU pt not in valid range (central three periods): kx[329]=9.469425314105727, N1=8000 (pirange=1)
FINUFFT invokeGuru: setpts error (ier=4)!
spreadcheck NU pt not in valid range (central three periods): kx[329]=9.469425314105727, N1=80